## 1. Cài đặt môi trường

### 1.1 Import thư viện

In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT MÔI TRƯỜNG & ĐỌC DỮ LIỆU
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score
import math

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {device}")

TRAIN_PATH = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/train.csv'
VAL_PATH   = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/val.csv'
TEST_PATH  = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/test.csv'

df_train_raw = pd.read_csv(TRAIN_PATH)
df_val_raw   = pd.read_csv(VAL_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

df_final_all = pd.concat([df_train_raw, df_val_raw, df_test_raw], ignore_index=True)
df_final_all['TIME'] = pd.to_datetime(df_final_all['TIME'])
df_final_all = df_final_all.sort_values('TIME').reset_index(drop=True)

n_train = len(df_train_raw)
n_val   = len(df_val_raw)
n_test  = len(df_test_raw)

print(f"Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,} | Total: {len(df_final_all):,}")

Thiết bị: cuda
Train: 82,675 | Val: 10,334 | Test: 10,335 | Total: 103,344


In [2]:
# ==============================================================================
# CELL 2: FEATURE ENGINEERING (ĐÃ FIX KHẮT KHE LỖI RÒ RỈ TƯƠNG LAI)
# ==============================================================================

# 1. Weather Forecasting (Dịch 48 bước = 24h)
weather_cols = ['temp', 'rhum', 'prcp', 'wspd']
forecast_weather_features = []
for col in weather_cols:
    new_col = f'{col}_forecast'
    df_final_all[new_col] = df_final_all[col].shift(-48)
    forecast_weather_features.append(new_col)

# 2. Cyclical Encoding
def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

df_final_all['Hour']      = df_final_all['TIME'].dt.hour
df_final_all['DayOfWeek'] = df_final_all['TIME'].dt.dayofweek
df_final_all['Month']     = df_final_all['TIME'].dt.month
df_final_all = encode_cyclical(df_final_all, 'Hour', 24)
df_final_all = encode_cyclical(df_final_all, 'DayOfWeek', 7)
df_final_all = encode_cyclical(df_final_all, 'Month', 12)
df_final_all['is_weekend'] = (df_final_all['DayOfWeek'] >= 5).astype(float)
cyclical_cols = ['Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'is_weekend']

# 3. Lag Features
lag_features = []
for l in [2, 4, 6, 12, 48, 336]:
    col = f'P224_lag{l}'
    df_final_all[col] = df_final_all['P_224'].shift(l)
    lag_features.append(col)

df_final_all['P224_roll24']  = df_final_all['P_224'].shift(1).rolling(24).mean()
df_final_all['P224_roll48']  = df_final_all['P_224'].shift(1).rolling(48).mean()
df_final_all['P224_std24']   = df_final_all['P_224'].shift(1).rolling(24).std()
df_final_all['P224_std48']   = df_final_all['P_224'].shift(1).rolling(48).std()
df_final_all['P224_delta1']  = df_final_all['P_224'].diff(1).shift(1)
df_final_all['P224_delta48'] = df_final_all['P_224'].diff(48).shift(1)
lag_features += ['P224_roll24', 'P224_roll48', 'P224_std24', 'P224_std48', 'P224_delta1', 'P224_delta48']

# =========================================================================
# 🔥 FIX LEAKAGE Ở ĐÂY: Xóa bfill(), dùng dropna() và bù trừ n_train
# =========================================================================
df_final_all = df_final_all.ffill() # Chỉ lấy QUÁ KHỨ lấp cho HIỆN TẠI (với các NaN nội bộ)
len_before = len(df_final_all)

# Cắt bỏ sạch sẽ phần đầu bị NaN do thiết lập Lag 336
df_final_all = df_final_all.dropna().reset_index(drop=True) 
rows_dropped = len_before - len(df_final_all)

# Cập nhật biến n_train (đã định nghĩa ở Cell 1) để Cell 3 cắt data khớp chính xác
n_train = n_train - rows_dropped

print(f"Đã dọn dẹp sạch dữ liệu: Cắt bỏ {rows_dropped} dòng mồ côi ở đầu để TỐI ĐA BẢO MẬT TƯƠNG LAI.")
assert df_final_all.isna().sum().sum() == 0, "Dữ liệu vẫn còn NaN!"
# =========================================================================

# 4. Gom nhóm
target_col = ['P_224']
substation_cols = [col for col in df_final_all.columns if col.startswith('P_') and col != 'P_224' and '_forecast' not in col]

df_final_all['Hour_float'] = df_final_all['TIME'].dt.hour + df_final_all['TIME'].dt.minute / 60.0
df_final_all['DayOfMonth'] = df_final_all['TIME'].dt.day
df_final_all['WeekOfYear'] = df_final_all['TIME'].dt.isocalendar().week.astype(int)
df_final_all['DayOfMonth_sin'] = np.sin(2 * np.pi * df_final_all['DayOfMonth'] / 31)
df_final_all['DayOfMonth_cos'] = np.cos(2 * np.pi * df_final_all['DayOfMonth'] / 31)
df_final_all['WeekOfYear_sin'] = np.sin(2 * np.pi * df_final_all['WeekOfYear'] / 52)
df_final_all['WeekOfYear_cos'] = np.cos(2 * np.pi * df_final_all['WeekOfYear'] / 52)
extra_time_cols = ['Hour_float', 'DayOfMonth_sin', 'DayOfMonth_cos', 'WeekOfYear_sin', 'WeekOfYear_cos']

base_features = forecast_weather_features + cyclical_cols + extra_time_cols + lag_features
topo_features = base_features + substation_cols

print("Hoàn tất Feature Engineering.")

Đã dọn dẹp sạch dữ liệu: Cắt bỏ 336 dòng mồ côi ở đầu để TỐI ĐA BẢO MẬT TƯƠNG LAI.
Hoàn tất Feature Engineering.


In [3]:
# ==============================================================================
# CELL 3: CẮT DATA & FIT SCALER (ĐÃ FIX LỖI MẤT DỮ LIỆU)
# ==============================================================================
WINDOW_SIZE = 48 

# Lùi lại WINDOW_SIZE dòng để lấy bối cảnh cho tập Val và Test
df_train = df_final_all.iloc[:n_train].copy()
df_val   = df_final_all.iloc[n_train - WINDOW_SIZE : n_train + n_val].copy()
df_test  = df_final_all.iloc[n_train + n_val - WINDOW_SIZE : ].copy()

# Fit Scaler (Chỉ trên Train)
scaler_X_base = StandardScaler()
scaler_X_topo = StandardScaler()
scaler_y      = RobustScaler()

scaler_X_base.fit(df_train[base_features])
scaler_X_topo.fit(df_train[topo_features])
scaler_y.fit(df_train[target_col])

print("Fit xong Scaler trên tập Train.")

Fit xong Scaler trên tập Train.


In [4]:
# ==============================================================================
# CELL 4: KIẾN TRÚC MÔ HÌNH (ĐÃ ĐỒNG BỘ GELU TOÀN CỤC)
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class KANLayer_Optimized(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3):
        super().__init__()
        self.in_features, self.out_features, self.grid_size, self.spline_order = in_features, out_features, grid_size, spline_order
        
        # 1. Đồng bộ GELU cho base activation của KAN
        self.base_activation = nn.GELU()
        
        self.base_weight   = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.spline_weight = nn.Parameter(torch.randn(out_features, in_features, grid_size + spline_order) * 0.1)
        self.register_buffer("grid", torch.linspace(-2.0, 2.0, grid_size + 1))

    def b_splines(self, x):
        x = x.unsqueeze(-1)
        h = (self.grid[1] - self.grid[0]).item()
        ext = torch.linspace(self.grid[0].item() - self.spline_order * h, self.grid[-1].item() + self.spline_order * h, self.grid_size + 1 + 2 * self.spline_order).to(x.device)
        view_shape = [1] * (x.dim() - 1) + [-1]
        ext = ext.view(*view_shape)
        bases = ((x >= ext[..., :-1]) & (x < ext[..., 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = ((x - ext[..., :-(k + 1)]) / (ext[..., k:-1] - ext[..., :-(k + 1)]) * bases[..., :-1] +
                     (ext[..., k + 1:] - x) / (ext[..., k + 1:] - ext[..., 1:-k]) * bases[..., 1:])
        return bases.contiguous()

    def update_grid(self, x, margin=0.01):
        with torch.no_grad():
            self.grid.copy_(torch.linspace(x.min().item() - margin, x.max().item() + margin, self.grid_size + 1).to(x.device))

    def forward(self, x):
        base_out = F.linear(self.base_activation(x), self.base_weight)
        if x.dim() == 2:
            spline_out = torch.einsum("big,oig->bo", self.b_splines(x), self.spline_weight)
        else:
            spline_out = torch.einsum("bsig,oig->bso", self.b_splines(x), self.spline_weight)
        return base_out + spline_out

class MultiHead_Vanilla_KAN(nn.Module):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64, fusion_dim=128, dropout=0.1, grid_size=5):
        super().__init__()
        self.flatten_dim = seq_len * num_features
        
        # 2. Đồng bộ GELU cho tầng Fusion nén dữ liệu
        self.fusion = nn.Sequential(
            nn.Linear(self.flatten_dim, fusion_dim), 
            nn.GELU(), 
            nn.Dropout(dropout)
        )
        
        self.ar_heads = nn.ModuleList([nn.Linear(fusion_dim, 1) for _ in range(output_dim)])
        self.layer1 = KANLayer_Optimized(fusion_dim, hidden_dim, grid_size=grid_size)
        self.layer2 = KANLayer_Optimized(hidden_dim, output_dim, grid_size=grid_size)
        self.layers = nn.ModuleList([self.layer1, self.layer2])

    def forward(self, x):
        fused = self.fusion(x.reshape(x.size(0), -1)) 
        ar_out = torch.cat([head(fused) for head in self.ar_heads], dim=1)
        return ar_out + self.layer2(self.layer1(fused))

    def regularization_loss(self, lamb_l1=0.01):
        return lamb_l1 * sum(layer.spline_weight.abs().mean() for layer in self.layers)

class SparseTopologyEncoder(nn.Module):
    def __init__(self, seq_len, num_features, topo_dim=32, num_layers=2, dropout=0.1, topk=4):
        super().__init__()
        self.num_features, self.topo_dim, self.topk = num_features, topo_dim, topk
        self.depthwise_conv = nn.Conv1d(num_features, num_features, kernel_size=3, padding=1, groups=num_features)
        self.temporal_proj = nn.Linear(seq_len, topo_dim)
        
        # 3. Đồng bộ GELU cho bộ mã hóa đồ thị Topo
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        
        self.msg_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(topo_dim, topo_dim), 
                nn.GELU(), # Cập nhật GELU tại đây
                nn.Dropout(dropout), 
                nn.Linear(topo_dim, topo_dim)
            ) for _ in range(num_layers)
        ])
        
        self.norms = nn.ModuleList([nn.LayerNorm(topo_dim) for _ in range(num_layers)])
        self.gates = nn.ModuleList([nn.Sequential(nn.Linear(topo_dim, topo_dim), nn.Sigmoid()) for _ in range(num_layers)])
        self.pool_proj = nn.Linear(topo_dim * 2, topo_dim)
        self.node_emb = nn.Parameter(torch.randn(num_features, topo_dim) * 0.1)

    def build_adj(self, device):
        sim = torch.matmul(self.node_emb, self.node_emb.t()) / math.sqrt(self.topo_dim)
        A = torch.relu(sim) + torch.eye(self.num_features, device=device)
        if self.topk < self.num_features:
            vals, idx = torch.topk(A, k=max(1, self.topk), dim=-1)
            mask = torch.zeros_like(A).scatter_(1, idx, 1.0)
            A = A * mask
        return A / A.sum(dim=-1, keepdim=True).clamp_min(1e-6)

    def forward(self, x):
        h = self.act(self.temporal_proj(self.depthwise_conv(x.transpose(1, 2))))
        A_b = self.build_adj(x.device).unsqueeze(0).expand(h.size(0), -1, -1)
        global_ctx = h.mean(dim=1, keepdim=True)
        for msg_mlp, gate, norm in zip(self.msg_mlps, self.gates, self.norms):
            h = norm(h + self.dropout(self.act(msg_mlp(torch.bmm(A_b, h) + global_ctx) * gate(h))))
        return self.pool_proj(torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1))

class Topo_KAN(nn.Module):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64, topo_dim=32, fusion_dim=128, dropout=0.1, grid_size=5):
        super().__init__()
        self.topo_encoder = SparseTopologyEncoder(seq_len, num_features, topo_dim=topo_dim, dropout=dropout)
        
        # 4. Đồng bộ GELU cho tầng Fusion của Topo-KAN
        self.fusion = nn.Sequential(
            nn.Linear(seq_len * num_features + topo_dim, fusion_dim), 
            nn.GELU(), 
            nn.Dropout(dropout)
        )
        
        self.ar_heads = nn.ModuleList([nn.Linear(fusion_dim, 1) for _ in range(output_dim)])
        self.kan_layer1 = KANLayer_Optimized(fusion_dim, hidden_dim, grid_size=grid_size)
        self.kan_layer2 = KANLayer_Optimized(hidden_dim, output_dim, grid_size=grid_size)
        self.topo_to_hidden = nn.Linear(topo_dim, hidden_dim)
        self.topo_to_out_gate = nn.Sequential(nn.Linear(topo_dim, output_dim), nn.Sigmoid())

    def forward(self, x):
        topo_vec = self.topo_encoder(x)
        fused = self.fusion(torch.cat([x.reshape(x.size(0), -1), topo_vec], dim=1))
        ar_out = torch.cat([head(fused) for head in self.ar_heads], dim=1)
        hidden = self.kan_layer1(fused) + self.topo_to_hidden(topo_vec)
        return ar_out + self.topo_to_out_gate(topo_vec) * self.kan_layer2(hidden)

    def regularization_loss(self, lamb_l1=0.01):
        return lamb_l1 * (self.kan_layer1.spline_weight.abs().mean() + self.kan_layer2.spline_weight.abs().mean())

print(" Đã nạp xong Mô hình với hàm GELU đồng bộ cho tất cả các tầng.")

 Đã nạp xong Mô hình với hàm GELU đồng bộ cho tất cả các tầng.


In [5]:
# ==============================================================================
# CELL 5: PIPELINE TRAIN & ĐÁNH GIÁ (ĐÃ FIX CHUẨN 30-PHÚT & FIX LỖI CRASH SCALER)
# ==============================================================================
def asymmetric_loss(y_pred, y_true, penalty_factor=3.0):
    error = y_true - y_pred
    loss = torch.where(error > 0, penalty_factor * (error ** 2), (error ** 2))
    return loss.mean()

def calibrate_kan_layers(model, calib_x):
    hooks = []
    def hook_fn(module, input, output):
        if hasattr(module, 'update_grid'): module.update_grid(input[0])
    for name, module in model.named_modules():
        if hasattr(module, 'update_grid'): hooks.append(module.register_forward_hook(hook_fn))
    with torch.no_grad(): model(calib_x)
    for h in hooks: h.remove()

def calculate_metrics(act, pre):
    eps = 1e-10
    mae = mean_absolute_error(act, pre)
    r2 = r2_score(act, pre)
    wape = np.sum(np.abs(act - pre)) / (np.sum(np.abs(act)) + eps) * 100
    return mae, wape, r2

def train_dynamic_model(model_obj, name, loader, val_loader, device, epochs=100):
    model_obj.eval()
    with torch.no_grad():
        calib_x, n = [], 0
        for bx, _ in loader:
            calib_x.append(bx.to(device))
            n += bx.size(0)
            if n >= 512: break
        calib_x = torch.cat(calib_x, dim=0)[:512]
        calibrate_kan_layers(model_obj, calib_x)

    optimizer = optim.Adam(model_obj.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    best_loss, counter, ckpt = float('inf'), 0, f"best_{name}.pth"

    for epoch in range(epochs):
        model_obj.train()
        train_losses = []
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model_obj(bx)
            loss = asymmetric_loss(pred, by, 3.0) + model_obj.regularization_loss(0.005)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_obj.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model_obj.eval()
        val_l = 0.0
        with torch.no_grad():
            for bx_v, by_v in val_loader:
                bx_v, by_v = bx_v.to(device), by_v.to(device)
                pred_v = model_obj(bx_v)
                val_l += asymmetric_loss(pred_v, by_v, 3.0).item()
        val_l /= len(val_loader)
        scheduler.step(val_l)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{name}] Epoch {epoch+1:03d}/{epochs} | Train: {np.mean(train_losses):.6f} | Val: {val_l:.6f}")

        if val_l < best_loss:
            best_loss, counter = val_l, 0
            torch.save(model_obj.state_dict(), ckpt)
        else:
            counter += 1
            
        if counter >= 15:
            print(f"  [{name}] Early stopping tại epoch {epoch+1}. Best Val Loss: {best_loss:.4f}")
            break

    model_obj.load_state_dict(torch.load(ckpt, map_location=device))
    return model_obj

def run_pipeline_for_horizon(forecast_steps, model_class, use_topo=False, epochs=100):
    mode_name = "TOPO" if use_topo else "BASE"
    # forecast_steps ở đây là số điểm (Ví dụ: truyền vào 6 là dự báo 6 điểm = 3 tiếng)
    print(f"\n{'='*75}")
    print(f" ĐANG TRAIN: {model_class.__name__} | MỐC: {forecast_steps} ĐIỂM ({(forecast_steps*0.5):.1f} GIỜ) | DỮ LIỆU: {mode_name}")
    print(f"{'='*75}")
    
    WINDOW_SIZE = 48 
    
    active_features = topo_features if use_topo else base_features
    active_scaler_X = scaler_X_topo if use_topo else scaler_X_base
    
    def create_custom_seq(df, s_x, s_y):
        X_raw = s_x.transform(df[active_features])
        y_raw = s_y.transform(df[target_col])
        X, y = [], []
        limit = len(df) - WINDOW_SIZE - forecast_steps + 1

        for i in range(limit):
            X_seq = X_raw[i : i + WINDOW_SIZE]
            # Lấy liên tiếp các điểm 30 phút
            y_seq = y_raw[i + WINDOW_SIZE : i + WINDOW_SIZE + forecast_steps].flatten()
            if len(y_seq) == forecast_steps:
                X.append(X_seq)
                y.append(y_seq)
        return torch.from_numpy(np.array(X, dtype=np.float32)), torch.from_numpy(np.array(y, dtype=np.float32))

    X_tr, y_tr = create_custom_seq(df_train, active_scaler_X, scaler_y)
    X_va, y_va = create_custom_seq(df_val,   active_scaler_X, scaler_y)
    X_te, y_te = create_custom_seq(df_test,  active_scaler_X, scaler_y)

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=64, shuffle=False)
    
    model = model_class(
        seq_len=WINDOW_SIZE, 
        num_features=len(active_features), 
        output_dim=forecast_steps, # Đầu ra khớp với số điểm
        hidden_dim=64, 
        fusion_dim=128,
        dropout=0.1,
        grid_size=5
    ).to(device)
    
    model_name = f"{model_class.__name__}_{forecast_steps}P"
    model = train_dynamic_model(model, model_name, train_loader, val_loader, device, epochs)
    
    model.eval()
    actuals, preds = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            preds.append(model(bx.to(device)).cpu().numpy())
            actuals.append(by.numpy())

    # =========================================================================
    # 🔥 ĐÃ FIX LỖI CRASH Ở ĐÂY: Duỗi thành 1 cột -> Giải mã -> Cuộn lại
    # =========================================================================
    flat_actuals = np.concatenate(actuals).reshape(-1, 1)
    flat_preds   = np.concatenate(preds).reshape(-1, 1)
    
    inv_act_flat = scaler_y.inverse_transform(flat_actuals)
    inv_pre_flat = scaler_y.inverse_transform(flat_preds)
    
    inv_act = inv_act_flat.reshape(-1, forecast_steps)
    inv_pre = inv_pre_flat.reshape(-1, forecast_steps)
    # =========================================================================
    
    # [ĐÃ FIX OVERLAP]: Cắt mảng chuẩn cho cấu trúc 30 phút/điểm
    step_jump = forecast_steps 
    act_non_overlap = inv_act[::step_jump, :].flatten()
    pre_non_overlap = inv_pre[::step_jump, :].flatten()
    
    mae, wape, r2 = calculate_metrics(act_non_overlap, pre_non_overlap)
    print(f" ĐÁNH GIÁ THỰC TẾ: WAPE: {wape:.2f}% | MAE: {mae:.2f} | R2: {r2:.4f}")
    
    return {"Mô hình": f"{model_class.__name__} ({mode_name})", "Mốc (Điểm)": forecast_steps, "Giờ tương ứng": f"{forecast_steps*0.5}H", "MAE": mae, "WAPE (%)": wape, "R2": r2}

In [6]:
# =========================================================================
# CELL 6: VÒNG LẶP CHẠY SO SÁNH (ĐÃ FIX TÊN THAM SỐ)
# =========================================================================

horizons = [1, 2, 6, 12, 24, 48] # Đây là số LƯỢNG ĐIỂM (Mỗi điểm 30p)

models_to_test = [
    (MultiHead_Vanilla_KAN, False),
    (Topo_KAN, True)
]

bang_tong_hop = []

for so_diem in horizons:
    for model_class, is_topo in models_to_test:
        try:
            set_seed(42)
            # GỌI ĐÚNG TÊN THAM SỐ: forecast_steps
            ket_qua = run_pipeline_for_horizon(
                forecast_steps=so_diem, 
                model_class=model_class,
                use_topo=is_topo,
                epochs=100
            )
            bang_tong_hop.append(ket_qua)
        except Exception as e:
            print(f"  Lỗi tại {model_class.__name__} - {so_diem}P: {e}")
            continue

# In bảng kết quả
if bang_tong_hop:
    df_report = pd.DataFrame(bang_tong_hop)
    df_report['WAPE (%)'] = df_report['WAPE (%)'].apply(lambda x: f"{x:.2f}%")
    df_report['MAE'] = df_report['MAE'].apply(lambda x: f"{x:.2f}")
    print("\n" + "="*30 + "\n BẢNG SO SÁNH KẾT QUẢ \n" + "="*30)
    print(df_report.sort_values(by=['Mốc (Điểm)', 'Mô hình']).to_string(index=False))


 ĐANG TRAIN: MultiHead_Vanilla_KAN | MỐC: 1 ĐIỂM (0.5 GIỜ) | DỮ LIỆU: BASE
  [MultiHead_Vanilla_KAN_1P] Epoch 001/100 | Train: 0.141459 | Val: 0.083846
  [MultiHead_Vanilla_KAN_1P] Epoch 005/100 | Train: 0.057373 | Val: 0.073434
  [MultiHead_Vanilla_KAN_1P] Epoch 010/100 | Train: 0.053238 | Val: 0.075158
  [MultiHead_Vanilla_KAN_1P] Epoch 015/100 | Train: 0.052682 | Val: 0.085841
  [MultiHead_Vanilla_KAN_1P] Epoch 020/100 | Train: 0.044125 | Val: 0.061370
  [MultiHead_Vanilla_KAN_1P] Epoch 025/100 | Train: 0.039913 | Val: 0.059880
  [MultiHead_Vanilla_KAN_1P] Epoch 030/100 | Train: 0.035976 | Val: 0.061951
  [MultiHead_Vanilla_KAN_1P] Epoch 035/100 | Train: 0.032702 | Val: 0.064334
  [MultiHead_Vanilla_KAN_1P] Epoch 040/100 | Train: 0.030609 | Val: 0.067509
  [MultiHead_Vanilla_KAN_1P] Early stopping tại epoch 40. Best Val Loss: 0.0599
 ĐÁNH GIÁ THỰC TẾ: WAPE: 10.06% | MAE: 2.04 | R2: 0.8934

 ĐANG TRAIN: Topo_KAN | MỐC: 1 ĐIỂM (0.5 GIỜ) | DỮ LIỆU: TOPO
  [Topo_KAN_1P] Epoch 001/100 |